In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('RetailDB').getOrCreate()


In [0]:
schema_text = spark.read.text('/Volumes/workspace/default/retail_data/schemas.json',
    wholetext=True
).first().value

In [0]:
import json

schema_data = json.loads(schema_text)

In [0]:
customers_column_details = schema_data['customers']
orders_column_details = schema_data['orders']
departments_column_details = schema_data['departments']
categories_column_details = schema_data['categories']
order_items_column_details = schema_data['order_items']
products_column_details = schema_data['products']

In [0]:
def load_table(table_name, column_details):

    columns = [col['column_name'] for col in column_details]

    df = spark.read.csv(
        f'/Volumes/workspace/default/retail_data/{table_name}.csv',
        inferSchema=True
    ).toDF(*columns)

    return df

In [0]:
customers = load_table('customers', customers_column_details)

orders = load_table('orders', orders_column_details)

departments = load_table('departments', departments_column_details)

categories = load_table('categories', categories_column_details)

order_items = load_table('order_item', order_items_column_details)

products = load_table('products', products_column_details)

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS retail_bronze


In [0]:
customers.write.format('delta').mode('overwrite').saveAsTable('retail_bronze.customers')

orders.write.format('delta').mode('overwrite').saveAsTable('retail_bronze.orders')

departments.write.format('delta').mode('overwrite').saveAsTable('retail_bronze.departments')

categories.write.format('delta').mode('overwrite').saveAsTable('retail_bronze.categories')

order_items.write.format('delta').mode('overwrite').saveAsTable('retail_bronze.order_items')

products.write.format('delta').mode('overwrite').saveAsTable('retail_bronze.products')